In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
class DataLoaderAndCleaner:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    def load_data(self):
        self.df = pd.read_excel(self.file_path)
        return self.df

    def inspect_data(self):
        print("Dataset Info")
        self.df.info()
        print("\n Missing Values")
        print(self.df.isnull().sum()[self.df.isnull().sum() > 0])
        print(f"\n Duplicate Rows: {self.df.duplicated().sum()} ")

     
data_path = "../data/raw/Sample - Superstore 2019.xls"


pipeline = DataLoaderAndCleaner(data_path)


pipeline.load_data()


pipeline.inspect_data()

print(pipeline.df.memory_usage(deep=True).sum() / 1024**2, "MB")

Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          9994 non-null   int64         
 1   Order ID        9994 non-null   object        
 2   Order Date      9994 non-null   datetime64[ns]
 3   Ship Date       9994 non-null   datetime64[ns]
 4   Ship Mode       9994 non-null   object        
 5   Customer ID     9994 non-null   object        
 6   Customer Name   9994 non-null   object        
 7   Segment         9994 non-null   object        
 8   Country/Region  9994 non-null   object        
 9   City            9994 non-null   object        
 10  State           9994 non-null   object        
 11  Postal Code     9983 non-null   float64       
 12  Region          9994 non-null   object        
 13  Product ID      9994 non-null   object        
 14  Category        9994 non-null   object     

In [3]:
df_clean = pipeline.df.copy()

if 'Postal Code' in df_clean.columns:
    df_clean['Postal Code'] = df_clean['Postal Code'].fillna(0).astype(int).astype(str)

df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])
df_clean['Ship Date'] = pd.to_datetime(df_clean['Ship Date'])

for col in df_clean.select_dtypes(include=['object']).columns:
    if df_clean[col].nunique() / len(df_clean) < 0.5:
        df_clean[col] = df_clean[col].astype('category')

for col in df_clean.select_dtypes(include=['float64']).columns:
    df_clean[col] = df_clean[col].astype('float32')



In [ ]:

class DataFrameWrapper:
    def __init__(self, df=None):
        self._df = df

    def load_data(self, file_path=None):
        if file_path:
            self._df = pd.read_excel(file_path)
        else:
            if self._df is not None:
                return self._df
            try:
                self._df = pipeline.load_data()
            except NameError:
                raise ValueError("No data available and no file_path provided")
        return self._df

    def inspect_data(self):
        df = self._df
        print(df.head())
        print("\nSummary:", type(df))
        print("\nDataset Info")
        df.info()
        print("\nMissing Values")
        print(df.isnull().sum()[df.isnull().sum() > 0])
        print(f"\nDuplicate Rows: {df.duplicated().sum()}")

    def __getattr__(self, name):
        return getattr(self._df, name)

# wrap the existing DataFrame so we can call the helper methods
df_clean = DataFrameWrapper(df_clean)

df_clean.load_data()

df_clean.inspect_data()

print("Memory usage:", df_clean.memory_usage(deep=True).sum() / 1024**2, "MB")


df_clean.inspect_data()

print("Memory usage:", df_clean.memory_usage(deep=True).sum() / 1024**2, "MB")

AttributeError: 'DataFrame' object has no attribute 'load_data'

In [18]:
os.makedirs("../data/processed", exist_ok=True)
df_clean.to_pickle("../data/processed/superstore_cleaned.pkl")